In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
OUTPUTS_DIR = Path("../Outputs")

# NEW emotion models (top 3)
emo_cardiff = pd.read_csv(OUTPUTS_DIR / "emotion_cardiff_twitter_905.csv")
emo_bertweet = pd.read_csv(OUTPUTS_DIR / "emotion_bertweet_905.csv")
emo_bhadresh = pd.read_csv(OUTPUTS_DIR / "emotion_bhadresh_distil_905.csv")

# Toxicity models (unchanged)
tox_detox = pd.read_csv(OUTPUTS_DIR / "toxicity_detoxify_905.csv")
tox_snlp = pd.read_csv(OUTPUTS_DIR / "toxicity_sNLP_905.csv")
tox_cardiff = pd.read_csv(OUTPUTS_DIR / "toxicity_cardiff_offensive_905.csv")

print(f"Cardiff Twitter emotion: {emo_cardiff.shape}")
print(f"BERTweet emotion:        {emo_bertweet.shape}")
print(f"bhadresh_distil emotion: {emo_bhadresh.shape}")
print(f"Detoxify:                {tox_detox.shape}")
print(f"s-nlp:                   {tox_snlp.shape}")
print(f"Cardiff Offensive:       {tox_cardiff.shape}")

Cardiff Twitter emotion: (905, 35)
BERTweet emotion:        (905, 31)
bhadresh_distil emotion: (905, 30)
Detoxify:                (905, 31)
s-nlp:                   (905, 26)
Cardiff Offensive:       (905, 26)


In [3]:
merged = pd.read_csv(OUTPUTS_DIR / "merged_text_905.csv")
master = merged[["student_id", "text_source", "has_text", "text_for_analysis"]].copy()
master["post_index"] = master.index

print(f"Master shape: {master.shape}")
print(f"Posts with text: {master['has_text'].sum()}")

Master shape: (905, 5)
Posts with text: 797


In [4]:
# 6 locked emotions
LOCKED_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# Cardiff has emo_ columns for its 11 labels — one-to-one for our 6
CARDIFF_MAP = {
    "anger": ["emo_anger"],
    "disgust": ["emo_disgust"],
    "fear": ["emo_fear"],
    "joy": ["emo_joy"],
    "sadness": ["emo_sadness"],
    "surprise": ["emo_surprise"],
}

# BERTweet has emo_ columns for 7 labels: joy, others, anger, surprise, sadness, fear, disgust
# Ignore "others" (it's basically neutral)
BERTWEET_MAP = {
    "anger": ["emo_anger"],
    "disgust": ["emo_disgust"],
    "fear": ["emo_fear"],
    "joy": ["emo_joy"],
    "sadness": ["emo_sadness"],
    "surprise": ["emo_surprise"],
}

# bhadresh_distil has emo_ columns for 6 labels: joy, love, anger, fear, surprise, sadness
# No disgust in bhadresh — will map as 0 for that emotion
# Love gets merged into joy
BHADRESH_MAP = {
    "anger": ["emo_anger"],
    "disgust": [],  # not in this model
    "fear": ["emo_fear"],
    "joy": ["emo_joy", "emo_love"],  # merge love into joy
    "sadness": ["emo_sadness"],
    "surprise": ["emo_surprise"],
}

print("Mappings defined")
print(f"Cardiff:  {sum(len(v) for v in CARDIFF_MAP.values())} labels → 6")
print(f"BERTweet: {sum(len(v) for v in BERTWEET_MAP.values())} labels → 6")
print(f"bhadresh: {sum(len(v) for v in BHADRESH_MAP.values())} labels → 6 (no disgust)")

Mappings defined
Cardiff:  6 labels → 6
BERTweet: 6 labels → 6
bhadresh: 6 labels → 6 (no disgust)


In [5]:
def collapse_emotions(df, mapping, model_prefix):
    """For each post, collapse source labels into 6 emotion scores using max."""
    result = pd.DataFrame(index=df.index)
    for our_emo, source_labels in mapping.items():
        valid = [c for c in source_labels if c in df.columns]
        if len(valid) == 0:
            result[f"{model_prefix}_{our_emo}"] = 0.0
        else:
            result[f"{model_prefix}_{our_emo}"] = df[valid].max(axis=1)
    return result

cardiff_emo = collapse_emotions(emo_cardiff, CARDIFF_MAP, "cardiff")
bertweet_emo = collapse_emotions(emo_bertweet, BERTWEET_MAP, "bertweet")
bhadresh_emo = collapse_emotions(emo_bhadresh, BHADRESH_MAP, "bhadresh")

# Drop existing emotion columns if any (safe re-run)
existing_emo = [c for c in master.columns if any(c.startswith(p) for p in ["cardiff_", "bertweet_", "bhadresh_"])
                and any(c.endswith(f"_{e}") for e in LOCKED_EMOTIONS)]
master = master.drop(columns=existing_emo)

# Join to master
master = master.join(cardiff_emo).join(bertweet_emo).join(bhadresh_emo)

print(f"Master shape after emotion mapping: {master.shape}")
print("\nSample first post:")
sample = master[master["has_text"]].iloc[0]
for emo in LOCKED_EMOTIONS:
    print(f"  {emo:10s}  cardiff={sample[f'cardiff_{emo}']:.2f}  bertweet={sample[f'bertweet_{emo}']:.2f}  bhadresh={sample[f'bhadresh_{emo}']:.2f}")

Master shape after emotion mapping: (905, 23)

Sample first post:
  anger       cardiff=0.77  bertweet=0.01  bhadresh=0.72
  disgust     cardiff=0.82  bertweet=0.38  bhadresh=0.00
  fear        cardiff=0.47  bertweet=0.00  bhadresh=0.23
  joy         cardiff=0.01  bertweet=0.01  bhadresh=0.02
  sadness     cardiff=0.23  bertweet=0.00  bhadresh=0.02
  surprise    cardiff=0.13  bertweet=0.00  bhadresh=0.00


In [6]:
for emo in LOCKED_EMOTIONS:
    cols = [f"cardiff_{emo}", f"bertweet_{emo}", f"bhadresh_{emo}"]
    master[f"emo_{emo}_max"] = master[cols].max(axis=1)
    master[f"emo_{emo}_min"] = master[cols].min(axis=1)
    master[f"emo_{emo}_range"] = master[f"emo_{emo}_max"] - master[f"emo_{emo}_min"]

# Overall emotion disagreement: max range across all 6 emotions
range_cols = [f"emo_{e}_range" for e in LOCKED_EMOTIONS]
master["emo_max_range"] = master[range_cols].max(axis=1)

print("Agreement metrics computed\n")
print("Average range per emotion (posts with text):")
for emo in LOCKED_EMOTIONS:
    avg = master[master["has_text"]][f"emo_{emo}_range"].mean()
    print(f"  {emo:10s}  avg range = {avg:.3f}")

print(f"\nDistribution of emo_max_range (overall disagreement):")
print(master[master["has_text"]]["emo_max_range"].describe())

Agreement metrics computed

Average range per emotion (posts with text):
  anger       avg range = 0.249
  disgust     avg range = 0.143
  fear        avg range = 0.142
  joy         avg range = 0.543
  sadness     avg range = 0.164
  surprise    avg range = 0.107

Distribution of emo_max_range (overall disagreement):
count    797.000000
mean       0.702271
std        0.284823
min        0.034670
25%        0.537559
50%        0.806761
75%        0.937072
max        0.997087
Name: emo_max_range, dtype: float64


In [7]:
tox_cols = ["tox_toxicity", "tox_severe_toxicity", "tox_obscene", "tox_identity_attack", "tox_insult", "tox_threat"]
detox_subset = tox_detox[tox_cols].copy()
detox_subset.columns = [f"detox_{c.replace('tox_', '')}" for c in tox_cols]

snlp_subset = tox_snlp[["snlp_toxic"]].copy()
cardiff_off_subset = tox_cardiff[["cardiff_offensive"]].copy()

# Drop existing tox columns if any
existing_tox = [c for c in master.columns if c.startswith(("detox_", "snlp_", "cardiff_offensive")) or c == "tox_range"]
master = master.drop(columns=existing_tox, errors="ignore")

master = master.join(detox_subset).join(snlp_subset).join(cardiff_off_subset)

master["tox_max"] = master[["detox_toxicity", "snlp_toxic", "cardiff_offensive"]].max(axis=1)
master["tox_min"] = master[["detox_toxicity", "snlp_toxic", "cardiff_offensive"]].min(axis=1)
master["tox_range"] = master["tox_max"] - master["tox_min"]

print(f"Master shape after toxicity merge: {master.shape}")

Master shape after toxicity merge: (905, 53)


In [8]:
RANGE_THRESHOLD = 0.25

def emotion_bucket(row):
    if not row["has_text"]:
        return "gray"
    text = str(row["text_for_analysis"])
    if len(text.strip()) < 20:
        return "gray"
    if row["emo_max_range"] > RANGE_THRESHOLD:
        return "yellow"
    return "green"

def toxicity_bucket(row):
    if not row["has_text"]:
        return "gray"
    text = str(row["text_for_analysis"])
    if len(text.strip()) < 20:
        return "gray"
    # Red flags
    if row["detox_threat"] > 0.5 or row["detox_identity_attack"] > 0.5 or row["detox_severe_toxicity"] > 0.5:
        return "red"
    # Broad disagreement
    if row["tox_range"] > RANGE_THRESHOLD:
        return "yellow"
    # Subtype-broad mismatch
    if row["detox_insult"] > 0.5 and row["snlp_toxic"] < 0.3:
        return "yellow"
    return "green"

master["emotion_bucket"] = master.apply(emotion_bucket, axis=1)
master["toxicity_bucket"] = master.apply(toxicity_bucket, axis=1)

print("=== Emotion bucket counts (v2 with new models) ===")
print(master["emotion_bucket"].value_counts())
print("\n=== Toxicity bucket counts ===")
print(master["toxicity_bucket"].value_counts())

=== Emotion bucket counts (v2 with new models) ===
emotion_bucket
yellow    653
gray      156
green      96
Name: count, dtype: int64

=== Toxicity bucket counts ===
toxicity_bucket
green     665
gray      156
yellow     83
red         1
Name: count, dtype: int64


In [9]:
# Load v1 for comparison
v1 = pd.read_csv(OUTPUTS_DIR / "routed_master_905.csv")

print("=== V1 (Cardiff + DistilRoBERTa + GoEmotions) ===")
print("Emotion buckets:")
print(v1["emotion_bucket"].value_counts())

print("\n=== V2 (Cardiff + BERTweet + bhadresh_distil) ===")
print("Emotion buckets:")
print(master["emotion_bucket"].value_counts())

# The improvement
v1_yellow = (v1["emotion_bucket"] == "yellow").sum()
v2_yellow = (master["emotion_bucket"] == "yellow").sum()
diff = v1_yellow - v2_yellow
pct = diff / v1_yellow * 100

print(f"\n=== Improvement ===")
print(f"V1 yellow: {v1_yellow}")
print(f"V2 yellow: {v2_yellow}")
print(f"Reduction: {diff} posts ({pct:.1f}%)")

=== V1 (Cardiff + DistilRoBERTa + GoEmotions) ===
Emotion buckets:
emotion_bucket
yellow    696
gray      156
green      53
Name: count, dtype: int64

=== V2 (Cardiff + BERTweet + bhadresh_distil) ===
Emotion buckets:
emotion_bucket
yellow    653
gray      156
green      96
Name: count, dtype: int64

=== Improvement ===
V1 yellow: 696
V2 yellow: 653
Reduction: 43 posts (6.2%)


In [11]:
output_path = OUTPUTS_DIR / "routed_master_v2_905.csv"
master.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {master.shape}")

Saved: ../Outputs/routed_master_v2_905.csv
Shape: (905, 55)


In [12]:
# Compare v1 vs v2 yellow buckets — see if any posts are yellow in v2 but weren't in v1
v1 = pd.read_csv(OUTPUTS_DIR / "routed_master_905.csv")
v2 = pd.read_csv(OUTPUTS_DIR / "routed_master_v2_905.csv")

v1_yellow_idx = set(v1.index[v1["emotion_bucket"] == "yellow"])
v2_yellow_idx = set(v2.index[v2["emotion_bucket"] == "yellow"])

# Posts that became yellow in v2 but weren't in v1 (need new LLM calls)
new_yellow = v2_yellow_idx - v1_yellow_idx
# Posts that were yellow in v1 but are green now in v2 (can use transformer scores)
new_green = v1_yellow_idx - v2_yellow_idx
# Posts yellow in both (LLM scores from v1 still valid)
still_yellow = v1_yellow_idx & v2_yellow_idx

print(f"V1 yellow: {len(v1_yellow_idx)}")
print(f"V2 yellow: {len(v2_yellow_idx)}")
print(f"")
print(f"Posts yellow in BOTH v1 and v2 (LLM scores reusable): {len(still_yellow)}")
print(f"Posts NEW yellow in v2 (need new LLM calls): {len(new_yellow)}")
print(f"Posts that moved v1-yellow → v2-green (use transformer): {len(new_green)}")

V1 yellow: 696
V2 yellow: 653

Posts yellow in BOTH v1 and v2 (LLM scores reusable): 608
Posts NEW yellow in v2 (need new LLM calls): 45
Posts that moved v1-yellow → v2-green (use transformer): 88


In [13]:
import os
import json
import time
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv("../.env")
client = Anthropic()
MODEL = "claude-sonnet-4-5"

# Emotion prompts (same as notebook 14)
EMOTION_OBJECTIVE = """You are analyzing a social media post for academic research.

Read this post and analyze its linguistic content. What emotions does the post EXPRESS? Score each of the 6 emotions from 0.0 to 1.0 based on the post's actual emotional tone, not just keywords. A post describing or reporting on an emotion is different from a post expressing it.

The 6 emotions:
- anger: hostility, rage, frustration
- disgust: revulsion, contempt, aversion
- fear: anxiety, dread, threat
- joy: happiness, delight, excitement
- sadness: grief, sorrow, melancholy
- surprise: shock, astonishment

Scoring: 0.0 = absent, 0.5 = moderately present, 1.0 = dominant. Use any decimal in between. Multiple emotions can co-exist.

Also rate your own confidence: how certain are you that your scores accurately capture this post's emotional content? Consider factors like text length, clarity, ambiguity. Be honest — 1.0 means very certain, 0.5 means moderately uncertain, 0.0 means basically guessing. Do not default to high confidence.

Post:
\"\"\"
{text}
\"\"\"

Respond with ONLY valid JSON in this exact format:
{{
  "anger": 0.0,
  "disgust": 0.0,
  "fear": 0.0,
  "joy": 0.0,
  "sadness": 0.0,
  "surprise": 0.0,
  "confidence": 0.0,
  "reasoning": "One sentence explaining the dominant emotion and your reasoning."
}}"""

EMOTION_YOUTH = """Imagine you ARE a 13-17 year old scrolling social media. You see this post on your feed. How would you FEEL?

Score each emotion from 0.0 to 1.0 based on your gut reaction as a young viewer.

The 6 emotions (how a teen would feel them):
- anger: irritated, mad, hostile
- disgust: grossed out, contemptuous, "ew"
- fear: anxious, worried, unsettled
- joy: happy, hyped, smiling, excited
- sadness: down, empathetically sad, blue
- surprise: shocked, "wait what", unexpected

Scoring: 0.0 = wouldn't feel it, 0.5 = moderate feeling, 1.0 = strong feeling. Use any decimal in between. Multiple feelings can occur at once.

Also rate your own confidence in your scores from 0.0 to 1.0. Be honest — do not default to high confidence.

Post you see:
\"\"\"
{text}
\"\"\"

Respond with ONLY valid JSON in this exact format:
{{
  "anger": 0.0,
  "disgust": 0.0,
  "fear": 0.0,
  "joy": 0.0,
  "sadness": 0.0,
  "surprise": 0.0,
  "confidence": 0.0,
  "reasoning": "One sentence explaining your gut reaction as a 13-17 year old viewer."
}}"""


def score_post(text, prompt_template, max_retries=3):
    text = str(text)[:2000]
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=400,
                temperature=1.0,
                messages=[{"role": "user", "content": prompt_template.format(text=text)}]
            )
            raw = response.content[0].text.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            parsed = json.loads(raw)
            if "confidence" not in parsed:
                parsed["confidence"] = None
            if "reasoning" not in parsed:
                parsed["reasoning"] = ""
            parsed["_input_tokens"] = response.usage.input_tokens
            parsed["_output_tokens"] = response.usage.output_tokens
            return parsed
        except json.JSONDecodeError as e:
            print(f"  JSON parse error attempt {attempt+1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"  API error attempt {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return None


# Load merged_text to get post content
merged = pd.read_csv(OUTPUTS_DIR / "merged_text_905.csv")

# Filter to just the new-yellow post indices
new_yellow_list = sorted(new_yellow)
new_yellow_posts = merged.loc[new_yellow_list].copy()
new_yellow_posts["_post_index"] = new_yellow_posts.index

print(f"Running LLM on {len(new_yellow_posts)} new-yellow posts × 2 prompts = {len(new_yellow_posts)*2} calls")
print(f"Estimated cost: ~${len(new_yellow_posts)*2*0.0034:.2f}")

# Run both prompts on each post
new_obj_results = []
new_youth_results = []

for idx, row in new_yellow_posts.iterrows():
    text = row["text_for_analysis"]
    
    # Objective
    obj = score_post(text, EMOTION_OBJECTIVE)
    if obj:
        obj["_post_index"] = idx
        obj["_student_id"] = row["student_id"]
        obj["_text_source"] = row["text_source"]
        new_obj_results.append(obj)
    
    # Youth
    youth = score_post(text, EMOTION_YOUTH)
    if youth:
        youth["_post_index"] = idx
        youth["_student_id"] = row["student_id"]
        youth["_text_source"] = row["text_source"]
        new_youth_results.append(youth)
    
    if len(new_obj_results) % 10 == 0:
        print(f"  Progress: {len(new_obj_results)}/{len(new_yellow_posts)}")

print(f"\nDone. Got {len(new_obj_results)} objective + {len(new_youth_results)} youth scores")

Running LLM on 45 new-yellow posts × 2 prompts = 90 calls
Estimated cost: ~$0.31
  Progress: 10/45
  Progress: 20/45
  Progress: 30/45
  Progress: 40/45

Done. Got 45 objective + 45 youth scores


In [14]:
# Append new results to existing LLM CSVs
existing_obj = pd.read_csv(OUTPUTS_DIR / "llm_emotion_objective_905.csv")
existing_youth = pd.read_csv(OUTPUTS_DIR / "llm_emotion_youth_905.csv")

new_obj_df = pd.DataFrame(new_obj_results)
new_youth_df = pd.DataFrame(new_youth_results)

combined_obj = pd.concat([existing_obj, new_obj_df], ignore_index=True)
combined_youth = pd.concat([existing_youth, new_youth_df], ignore_index=True)

# Save (overwrites existing files with expanded versions)
combined_obj.to_csv(OUTPUTS_DIR / "llm_emotion_objective_905.csv", index=False)
combined_youth.to_csv(OUTPUTS_DIR / "llm_emotion_youth_905.csv", index=False)

print(f"Updated llm_emotion_objective_905.csv: {len(combined_obj)} rows")
print(f"Updated llm_emotion_youth_905.csv: {len(combined_youth)} rows")

Updated llm_emotion_objective_905.csv: 741 rows
Updated llm_emotion_youth_905.csv: 741 rows
